# 🧠 Taranga AI: Technical Report — Multi-Output ML Pipeline

**Module:** `02_model_trainer`  
**Objective:** Train and benchmark a high-performance multi-label classifier for identifying LD risk profiles.

---

## 1. Executive Summary

The Taranga screening model is a **Multi-Output Classifier**. Unlike traditional classifiers that predict a single category, our model treats each of the 5 learning difficulty (LD) domains as an independent target variable. This architecture allows for the identification of **co-occurring conditions** (comorbidity), which is critical for accurate student support.

### 1.1 Benchmarking Methodology
We compared two industry-standard ensemble algorithms:
1. **Random Forest (RF):** 200 trees, focusing on lowering variance through bagging.
2. **Gradient Boosting (GB):** 200 estimators, focusing on lowering bias through sequential learning.

---

## 2. Training Infrastructure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, hamming_loss, f1_score
import joblib, os

sns.set_theme(style="whitegrid", palette="pastel")

FEATURE_COLS = [
    "q1_reading_pace", "q2_spelling_errors", "q3_letter_reversal", "q4_phonological",
    "q5_math_operations", "q6_number_memory", "q7_time_concept", "q8_counting_patterns",
    "q9_writing_quality", "q10_pencil_grip", "q11_word_spacing", "q12_copying_accuracy",
    "q13_social_cues", "q14_spatial_reasoning", "q15_routine_flexibility", "q16_nonverbal_comprehension",
    "q17_background_noise", "q18_verbal_instructions", "q19_sound_discrimination", "q20_listening_retention",
]
LABEL_COLS = ["dyslexia", "dyscalculia", "dysgraphia", "nvld", "apd"]

data_path = "../backend/data/synthetic_dataset.csv"
df = pd.read_csv(data_path)
X = df[FEATURE_COLS]
y = df[LABEL_COLS]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Performance Metrics

### 3.1 Model Benchmarking: RF vs GB
We evaluate the models using **Hamming Loss** (the fraction of labels that are incorrectly predicted) and the **Micro F1-Score**.

In [ ]:
rf = MultiOutputClassifier(RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)).fit(X_train, y_train)
gb = MultiOutputClassifier(GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42)).fit(X_train, y_train)

rf_hl = hamming_loss(y_test, rf.predict(X_test))
gb_hl = hamming_loss(y_test, gb.predict(X_test))
rf_f1 = f1_score(y_test, rf.predict(X_test), average='micro')
gb_f1 = f1_score(y_test, gb.predict(X_test), average='micro')

results = pd.DataFrame({
    "Metric": ["Hamming Loss (↓)", "Micro F1-Score (↑)"],
    "Random Forest": [rf_hl, rf_f1],
    "Gradient Boosting": [gb_hl, gb_f1]
})

plt.figure(figsize=(10, 5))
results.set_index("Metric").plot(kind='bar', color=['#6366F1', '#10B981'])
plt.title("Model Comparison: Classifier Benchmarking", fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.legend(loc='lower center')
plt.show()

### 3.2 Detailed Classification Report
Below is the per-domain performance report for the chosen production model (**RandomForest**).

In [ ]:
from sklearn.metrics import classification_report
report_dict = classification_report(y_test, rf.predict(X_test), target_names=LABEL_COLS, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose().iloc[:5, :3]

plt.figure(figsize=(12, 6))
sns.heatmap(report_df, annot=True, cmap="YlGnBu", cbar=False, fmt=".2f")
plt.title("Classification Performance Map (Precision / Recall / F1)", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Standard Text-Based Classification Report
print("### Standard Classification Report ###\n")
print(classification_report(y_test, rf.predict(X_test), target_names=LABEL_COLS))

### 3.3 Binary Confusion Matrices
We use binary confusion matrices to verify the precise True Positive, True Negative, False Positive, and False Negative counts for each specific LD domain.

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix

mcm = multilabel_confusion_matrix(y_test, rf.predict(X_test))

plt.figure(figsize=(18, 10))
for i, ld in enumerate(LABEL_COLS):
    plt.subplot(2, 3, i+1)
    sns.heatmap(mcm[i], annot=True, fmt="d", cmap="Blues", cbar=False, 
                xticklabels=['Pred Neg', 'Pred Pos'], 
                yticklabels=['Actual Neg', 'Actual Pos'])
    plt.title(f"Confusion Matrix: {ld.upper()}", fontsize=12, fontweight='bold')

plt.suptitle("Binary Confusion Matrices per LD Domain", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 4. Feature Importance Analysis

Feature importance allows us to understand exactly which screening questions "trigger" the model to identify a specific LD. We analyze the top predictors for each primary domain.

In [ ]:
plt.figure(figsize=(18, 12))
for i, ld in enumerate(LABEL_COLS):
    plt.subplot(2, 3, i+1)
    importances = rf.estimators_[i].feature_importances_
    indices = np.argsort(importances)[-8:] # Top 8
    plt.barh(range(len(indices)), importances[indices], color=sns.color_palette("viridis", 8))
    plt.yticks(range(len(indices)), [FEATURE_COLS[j] for j in indices])
    plt.title(f"{ld.upper()} Predictors", fontsize=12, fontweight='bold')

plt.suptitle("Feature Importance: Dominant Diagnostic Indicators per LD Domain", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 5. Model Serialization
The model is exported to the `models/` directory for use by the backend API and the SHAP explainer module.

In [ ]:
os.makedirs("../backend/app/ai/models", exist_ok=True)
save_path = "../backend/app/ai/models/ml_model.joblib"
joblib.dump(rf, save_path)
print(f"✅ Production Model successfully serialized to: {save_path}")